# ANON AI ULTRA — İlk Türkçe modeli sıfırdan eğit

Bu deney, önceden eğitilmiş herhangi bir model ağırlığı indirmez. Tokenizer ve yaklaşık 30–35M parametreli GPT tarzı model rastgele başlangıç ağırlıklarıyla oluşturulur. Bu ilk çıktı yalnızca altyapı denemesidir; henüz konuşma asistanı değildir.

**Kaggle veya Google Colab:** Notebook ayarlarından GPU seç ve Internet erişimini aç. Ücretsiz GPU erişimi garantili değildir.

In [ ]:
import os, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU bulunamadı')
assert torch.cuda.is_available(), 'Notebook ayarlarından GPU seç ve tekrar çalıştır.'
WORK_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/content'
print('Çalışma klasörü:', WORK_DIR)

In [ ]:
import os, subprocess, sys
REPO_DIR = os.path.join(WORK_DIR, 'ANON-AI-ULTRA')
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/furkanmeyra2173-create/ANON-AI-ULTRA.git', REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'training/requirements.txt'], check=True)

## Eğitim

İlk denemede 30.000 belge ve 500 optimizer adımı kullanılır. Eğitim süresi GPU'ya ve veri indirme hızına bağlıdır. Kesinti olursa bu ilk sürüm veriyi yeniden hazırlayarak başlar; otomatik devam ettirme henüz eklenmedi.

In [ ]:
import os, subprocess, sys
OUTPUT_DIR = os.path.join(WORK_DIR, 'anon-ai-ultra-base-30m')
command = [
    sys.executable, 'training/train_from_scratch.py',
    '--dataset-id', 'Ethosoft/Turkish_corpus',
    '--split', 'train',
    '--text-column', 'text',
    '--max-documents', '30000',
    '--max-steps', '500',
    '--output-dir', OUTPUT_DIR,
]
subprocess.run(command, check=True)

## Model çıktısını paketle

Eğitim başarıyla tamamlanırsa ZIP dosyasını Kaggle'ın Output bölümünden indirebilirsin. Android uygulamasına henüz bağlama; önce model kalitesini değerlendireceğiz.

In [ ]:
from pathlib import Path
import zipfile
base = Path(OUTPUT_DIR)
final_dir = base / 'final'
assert final_dir.exists(), 'Model klasörü bulunamadı; eğitim hücresinin hatasız tamamlandığını kontrol et.'
zip_path = Path(WORK_DIR) / 'ANON-AI-ULTRA-base-30m.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for file in final_dir.rglob('*'):
        if file.is_file():
            archive.write(file, file.relative_to(base))
    metadata = base / 'training_metadata.json'
    if metadata.exists():
        archive.write(metadata, metadata.name)
print('Hazır:', zip_path, 'Boyut (MB):', round(zip_path.stat().st_size / 1024**2, 2))